In [ ]:
# Escondendo Warnings
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# -*- coding: utf-8 -*-
from pathlib import Path
from io import BytesIO
import gc

import numpy as np
import pandas as pd
from typing import List
import pyarrow.parquet as pq

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator

from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.pdfgen.canvas import Canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfbase.pdfmetrics import registerFontFamily
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    PageBreak,
    Image,
    Table,
    TableStyle,
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle


# ============================================================
# I/O (ALTERAÇÃO NECESSÁRIA: leitura em batches para não estourar RAM)
# ============================================================
BATCH_SIZE = 250_000  # ajuste (300k–1M) conforme sua RAM

def _schema_cols(path: Path) -> set[str]:
    return set(pq.read_schema(str(path)).names)

def _pick_col(path: Path, candidates: list[str]) -> str | None:
    cols = _schema_cols(path)
    for c in candidates:
        if c in cols:
            return c
    return None

def _iter_batches(path: Path, columns: list[str], batch_size: int = BATCH_SIZE):
    pf = pq.ParquetFile(str(path))
    for batch in pf.iter_batches(columns=columns, batch_size=batch_size):
        yield batch.to_pandas()

def _read_parquet_cols_safe(path: Path, cols: list[str]) -> pd.DataFrame:
    schema_cols = set(pq.read_schema(str(path)).names)
    use = [c for c in cols if c in schema_cols]
    return pq.read_table(str(path), columns=use).to_pandas()


# ============================================================
# Configurações
# ============================================================
prefixo_area = "B-ROT"
col_irregularidade = "Irregularidade - 999 como Trajeto"

# >>> semanal (relatório geral)
freq = "W-MON"

# >>> mensal (somente para os 2 últimos gráficos - Top 2)
freq_top2 = "MS"  # Month Start (mensal)

max_pessoas_graficos = None

PARQUET_DADOS = Path(r".\dados.parquet")
PARQUET_RONDA = Path(r".\df_ronda.parquet")

# manter sempre este cadastro, mesmo fora do B-ROT*
CADASTRO_FORCADO = "2069687"

base_dir = Path(__file__).parent if "__file__" in globals() else Path.cwd()
out_dir = base_dir / "saidas_relatorio"
out_dir.mkdir(parents=True, exist_ok=True)

pdf_path = out_dir / f"Relatorio_Auditoria_Irregularidade_999_{prefixo_area}.pdf"
excel_path = out_dir / f"Linhas_Irregularidade_999_{prefixo_area}.xlsx"

data_geracao_relatorio = pd.Timestamp.now().strftime("%d/%m/%Y")


# ============================================================
# Parâmetros da regra
# ============================================================
janela_999_horas = 6
max_jornada_portaria_horas = 20
hora_corte_operacional = 11

t_jan_999 = pd.Timedelta(hours=janela_999_horas)
t_max_jornada_port = pd.Timedelta(hours=max_jornada_portaria_horas)

regex_portaria = r"\bPORT(?:ARIA)?\b|\bCANCELA\b|\bCANCELA\s*-\s*PORT\b|\bCANCELA VEICULO\s+[A-Z0-9]+\b"


# ============================================================
# Fonte Segoe UI (opcional)
# ============================================================
def _try_register_vw_text(fonts_dir: Path):
    regular_candidates = [
        fonts_dir / "CorpText-Regular.ttf",
        fonts_dir / "CorpText.ttf",
        fonts_dir / "CORP_Text.ttf",
        fonts_dir / "CorpText-Regular.otf",
        fonts_dir / "CorpText.otf",
    ]
    bold_candidates = [
        fonts_dir / "CorpText-Bold.ttf",
        fonts_dir / "CorpTextBold.ttf",
        fonts_dir / "CORP_Text_Bold.ttf",
        fonts_dir / "CorpText-Bold.otf",
        fonts_dir / "CorpTextBold.otf",
    ]

    reg_path = next((p for p in regular_candidates if p.exists()), None)
    bold_path = next((p for p in bold_candidates if p.exists()), None)

    if reg_path is None:
        return ("Helvetica", "Helvetica-Bold")

    pdfmetrics.registerFont(TTFont("CorpText", str(reg_path)))

    if bold_path is not None:
        pdfmetrics.registerFont(TTFont("CorpText-Bold", str(bold_path)))
        registerFontFamily(
            "CorpText",
            normal="CorpText",
            bold="CorpText-Bold",
            italic="CorpText",
            boldItalic="CorpText-Bold",
        )
        return ("CorpText", "CorpText-Bold")

    registerFontFamily("CorpText", normal="CorpText", bold="CorpText", italic="CorpText", boldItalic="CorpText")
    return ("CorpText", "CorpText")


FONT_REG, FONT_BOLD = _try_register_vw_text(Path("relatorios") / "fonts")


# ============================================================
# Helpers
# ============================================================
def br_int(n: int) -> str:
    return f"{int(n):,}".replace(",", ".")

def br_pct(x: float) -> str:
    return f"{x*100:.2f}%".replace(".", ",")

def _norm_cadastro(series: pd.Series) -> pd.Series:
    s = series.copy()
    if pd.api.types.is_numeric_dtype(s):
        s = pd.to_numeric(s, errors="coerce").astype("Int64").astype(str)
    else:
        s = s.astype(str)
    s = s.str.strip()
    s = s.replace({"<NA>": "", "nan": "", "None": "", "NaT": ""})
    s = s.str.replace(r"\.0$", "", regex=True)
    return s

def _fix_mojibake(s: pd.Series) -> pd.Series:
    x = s.fillna("").astype(str)
    m = x.str.contains(r"[ÃÂï¿½]", regex=True, na=False)
    if not m.any():
        return x

    def _try(v: str) -> str:
        try:
            return v.encode("latin1", errors="ignore").decode("utf-8", errors="ignore")
        except Exception:
            return v

    x = x.where(~m, x.map(_try))
    return x

def _normalize_hora_col(series: pd.Series) -> pd.Series:
    if series is None:
        return series
    s = series.copy()

    if pd.api.types.is_datetime64_any_dtype(s):
        return s.dt.strftime("%H:%M:%S")

    if pd.api.types.is_timedelta64_dtype(s):
        total_seconds = (s / np.timedelta64(1, "s")).round().astype("Int64")
        hh = (total_seconds // 3600).astype("Int64")
        mm = ((total_seconds % 3600) // 60).astype("Int64")
        ss = (total_seconds % 60).astype("Int64")
        return (
            hh.astype(str).str.zfill(2) + ":" +
            mm.astype(str).str.zfill(2) + ":" +
            ss.astype(str).str.zfill(2)
        )

    s = s.fillna("").astype(str).str.strip()
    s = s.replace({"nan": "", "NaT": "", "None": ""})

    m_hhmm = s.str.match(r"^\d{1,2}:\d{2}$", na=False)
    s = s.where(~m_hhmm, s + ":00")

    m_h = s.str.match(r"^\d:\d{2}:\d{2}$", na=False)
    s = s.where(~m_h, s.str.replace(r"^(\d):", r"0\1:", regex=True))

    return s

def _norm_dir(series: pd.Series) -> pd.Series:
    s = series.fillna("").astype(str).str.strip().str.upper()
    s = s.replace({"ENTRADA": "E", "SAIDA": "S", "SAÍDA": "S"})
    s = s.where(~s.str.startswith("E"), "E")
    s = s.where(~s.str.startswith("S"), "S")
    return s

# cria mapa cadastro -> area a partir do df_ronda.parquet (ronda_base)
def _build_area_map_from_ronda(ronda_df: pd.DataFrame, prefixo: str) -> dict[str, str]:
    """
    Cria um mapa cadastro -> area usando SOMENTE os registros vindos do df_ronda.parquet (ronda_base).
    Regra:
      - escolhe a área mais frequente por cadastro
      - em empate, prioriza áreas que começam com o prefixo (ex.: B-ROT)
    """
    if ronda_df is None or ronda_df.empty:
        return {}

    d = ronda_df[["cadastro", "area"]].copy()
    d["cadastro"] = _norm_cadastro(d["cadastro"])
    d["area"] = d["area"].fillna("").astype(str).str.strip()

    d = d[(d["cadastro"].ne("")) & (d["area"].ne(""))]
    if d.empty:
        return {}

    counts = (
        d.groupby(["cadastro", "area"], sort=False)
         .size()
         .rename("n")
         .reset_index()
    )
    counts["_pref"] = counts["area"].str.startswith(prefixo, na=False).astype(int)

    counts = counts.sort_values(
        ["cadastro", "_pref", "n", "area"],
        ascending=[True, False, False, True],
        kind="mergesort",
    )

    best = counts.drop_duplicates("cadastro", keep="first").set_index("cadastro")["area"].to_dict()
    return best


# ============================================================
# Curva nos gráficos: spline cúbica natural (sem scipy)
# ============================================================
def _natural_cubic_spline(x: np.ndarray, y: np.ndarray, x_new: np.ndarray) -> np.ndarray:
    n = len(x)
    if n < 3:
        return np.interp(x_new, x, y)

    h = np.diff(x)
    if np.any(h == 0):
        return np.interp(x_new, x, y)

    alpha = np.zeros(n)
    for i in range(1, n - 1):
        alpha[i] = (3 / h[i]) * (y[i + 1] - y[i]) - (3 / h[i - 1]) * (y[i] - y[i - 1])

    l = np.ones(n)
    mu = np.zeros(n)
    z = np.zeros(n)

    for i in range(1, n - 1):
        l[i] = 2 * (x[i + 1] - x[i - 1]) - h[i - 1] * mu[i - 1]
        mu[i] = h[i] / l[i]
        z[i] = (alpha[i] - h[i - 1] * z[i - 1]) / l[i]

    b = np.zeros(n - 1)
    c = np.zeros(n)
    d = np.zeros(n - 1)
    a = y.copy()

    for j in range(n - 2, -1, -1):
        c[j] = z[j] - mu[j] * c[j + 1]
        b[j] = ((a[j + 1] - a[j]) / h[j]) - (h[j] * (c[j + 1] + 2 * c[j]) / 3)
        d[j] = (c[j + 1] - c[j]) / (3 * h[j])

    idx = np.searchsorted(x, x_new, side="right") - 1
    idx = np.clip(idx, 0, n - 2)
    dx = x_new - x[idx]
    y_new = a[idx] + b[idx] * dx + c[idx] * (dx ** 2) + d[idx] * (dx ** 3)
    return y_new

def _ajustar_anotacoes_sem_sobrepor(fig, ax, anotacoes: List, pad=1.06, dx=6, dy=6, max_iter=200):
    if len(anotacoes) < 2:
        return

    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    for _ in range(max_iter):
        moved = False
        bboxes = [t.get_window_extent(renderer).expanded(pad, pad) for t in anotacoes]

        for i in range(len(anotacoes)):
            for j in range(i):
                if bboxes[i].overlaps(bboxes[j]):
                    xoff, yoff = anotacoes[i].get_position()
                    xoff += dx if (i % 2 == 0) else -dx
                    yoff += dy
                    anotacoes[i].set_position((xoff, yoff))
                    moved = True

        if not moved:
            break

        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()

def plot_linha_para_buffer(x, y, titulo, ylabel, dias_com=None, ymax=None, somente_linha: bool = False) -> BytesIO:
    x_ser = pd.to_datetime(pd.Series(x), errors="coerce")
    m = x_ser.notna()
    x_ser = x_ser[m]

    y_ser = pd.Series(y).reindex(x_ser.index).fillna(0).astype(int)

    d_ser = None
    if dias_com is not None:
        d_ser = pd.Series(dias_com).reindex(x_ser.index).fillna(0).astype(int)

    x_dt = x_ser.to_numpy()
    y_arr = y_ser.to_numpy()

    fig, ax = plt.subplots(figsize=(7.0, 3.9))

    if len(x_dt) >= 3:
        x_num = mdates.date2num(x_dt)
        k = max(150, (len(x_num) - 1) * 25)
        x_dense = np.linspace(x_num.min(), x_num.max(), k)
        y_dense = _natural_cubic_spline(x_num, y_arr.astype(float), x_dense)

        if ymax is not None:
            y_dense = np.clip(y_dense, 0, ymax)
        else:
            y_dense = np.clip(y_dense, 0, None)

        ax.plot(
            mdates.num2date(x_dense),
            y_dense,
            linewidth=3.0,
            solid_capstyle="round",
            solid_joinstyle="round",
        )

        if not somente_linha:
            ax.plot(
                x_dt,
                y_arr,
                marker="o",
                linewidth=0,
                markersize=6.0,
                markerfacecolor="white",
                markeredgewidth=2.0,
            )
            ax.fill_between(mdates.num2date(x_dense), y_dense, 0, alpha=0.10)
    else:
        y_plot = y_arr.copy()
        if ymax is not None:
            y_plot = np.clip(y_plot, 0, ymax)

        ax.plot(
            x_dt,
            y_plot,
            marker=None if somente_linha else "o",
            linewidth=3.0,
            markersize=0 if somente_linha else 6.0,
            markerfacecolor="white",
            markeredgewidth=2.0,
            solid_capstyle="round",
            solid_joinstyle="round",
        )
        if not somente_linha:
            ax.fill_between(x_dt, y_plot, 0, alpha=0.10)

    if not somente_linha:
        anotacoes = []
        for i, (xi, yi) in enumerate(zip(x_dt, y_arr)):
            if yi == 0:
                continue

            label = f"{int(yi)}"
            xoff = -4 if (i % 2 == 0) else 4
            yoff = 4 + (i % 3) * 1.5

            ann = ax.annotate(
                label,
                (xi, min(yi, ymax) if ymax is not None else yi),
                textcoords="offset points",
                xytext=(xoff, yoff),
                ha="center",
                va="bottom",
                fontsize=8,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75),
            )
            anotacoes.append(ann)

        _ajustar_anotacoes_sem_sobrepor(fig, ax, anotacoes, pad=1.08, dx=6, dy=6, max_iter=200)

    ax.grid(True, linestyle="--", linewidth=0.7, alpha=0.6)
    ax.set_title(titulo, fontsize=11)
    ax.set_xlabel("Período")
    ax.set_ylabel(ylabel)

    if ymax is not None:
        ax.set_ylim(0, ymax)
    else:
        ax.set_ylim(bottom=0)

    ax.margins(y=0.18)

    locator = mdates.AutoDateLocator()
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=220, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return buf


# ============================================================
# 0) Carrega e padroniza (leitura em batches + filtra cedo)
# ============================================================
def _collect_cadastros_prefixo_area_ronda(path: Path, prefixo: str) -> set[str]:
    area_col = _pick_col(path, ["area", "filial"])
    cad_col = _pick_col(path, ["cadastro"])

    if area_col is None or cad_col is None:
        raise RuntimeError(f"Não achei colunas esperadas em {path.name}. area_col={area_col} cad_col={cad_col}")

    cad_set: set[str] = set()
    for df in _iter_batches(path, [area_col, cad_col]):
        area = df[area_col].astype(str)
        m = area.str.startswith(prefixo, na=False)
        if not m.any():
            continue
        cad = _norm_cadastro(df.loc[m, cad_col])
        cad = cad[cad.ne("")]
        cad_set.update(cad.tolist())

    return cad_set

def _load_ronda_filtrado(path: Path, cad_set: set[str]) -> pd.DataFrame:
    cols = _schema_cols(path)
    area_src = "area" if "area" in cols else ("filial" if "filial" in cols else None)

    use_cols = [c for c in [
        "cadastro", "nome", area_src, "data", "hora", "dir", "planta", "coletor", "descricao_coletor", "cargo"
    ] if c and c in cols]

    chunks = []
    for df in _iter_batches(path, use_cols):
        df["cadastro"] = _norm_cadastro(df["cadastro"])
        m = df["cadastro"].isin(cad_set)
        if not m.any():
            continue

        d = df.loc[m].copy()
        if area_src and area_src in d.columns and area_src != "area":
            d = d.rename(columns={area_src: "area"})
        if "area" not in d.columns:
            d["area"] = pd.Series([""] * len(d), index=d.index)

        if "descricao_coletor" not in d.columns:
            d["descricao_coletor"] = ""
        if "cargo" not in d.columns:
            d["cargo"] = ""
        if "nome" not in d.columns:
            d["nome"] = ""

        d["_src_prio"] = 0
        chunks.append(d[["area", "cadastro", "nome", "dir", "planta", "coletor", "descricao_coletor", "data", "hora", "cargo", "_src_prio"]])

    if not chunks:
        return pd.DataFrame(columns=["area", "cadastro", "nome", "dir", "planta", "coletor", "descricao_coletor", "data", "hora", "cargo", "_src_prio"])
    return pd.concat(chunks, ignore_index=True)

def _load_dados_filtrado(path: Path, cad_set: set[str]) -> pd.DataFrame:
    col_area = _pick_col(path, ["Filial", "filial", "Area", "AREA", "area", "Área", "área", "Unidade", "Local"])
    col_cad = _pick_col(path, ["Cadastro", "cadastro"])
    col_nome = _pick_col(path, ["Nome", "nome"])
    col_dir = _pick_col(path, ["Dir", "dir"])
    col_planta = _pick_col(path, ["Planta", "planta"])
    col_coletor = _pick_col(path, ["Coletor", "coletor"])
    col_data = _pick_col(path, ["Data", "data"])
    col_hora = _pick_col(path, ["Hora", "hora"])

    desc_col = _pick_col(path, ["Descrição Coletor", "DescriÃ§Ã£o Coletor", "Descricao Coletor"])
    col_acesso = _pick_col(path, ["Acesso", "acesso"])
    col_tipo = _pick_col(path, ["Tipo Acesso", "tipo acesso", "TipoAcesso"])

    if col_cad is None:
        raise RuntimeError(f"Não achei coluna Cadastro em {path.name}")

    use_cols = [c for c in [
        col_area, col_cad, col_nome, col_dir, col_planta, col_coletor,
        col_data, col_hora, desc_col, col_acesso, col_tipo
    ] if c is not None]

    chunks = []
    for df in _iter_batches(path, use_cols):
        df[col_cad] = _norm_cadastro(df[col_cad])
        m = df[col_cad].isin(cad_set)
        if not m.any():
            continue

        d0 = df.loc[m].copy()

        coletor_raw = d0[col_coletor] if (col_coletor and col_coletor in d0.columns) else pd.Series([None] * len(d0), index=d0.index)
        acesso_raw = d0[col_acesso] if (col_acesso and col_acesso in d0.columns) else pd.Series([None] * len(d0), index=d0.index)
        coletor_num = pd.to_numeric(coletor_raw, errors="coerce")
        acesso_num = pd.to_numeric(acesso_raw, errors="coerce")
        usar_acesso_como_coletor = coletor_num.isna() & acesso_num.notna()
        coletor_eff = coletor_raw.where(~usar_acesso_como_coletor, acesso_raw)

        if desc_col and desc_col in d0.columns:
            desc_eff = d0[desc_col]
        else:
            a = d0[col_acesso] if (col_acesso and col_acesso in d0.columns) else pd.Series([""] * len(d0), index=d0.index)
            t = d0[col_tipo] if (col_tipo and col_tipo in d0.columns) else pd.Series([""] * len(d0), index=d0.index)
            desc_eff = (a.fillna("").astype(str).str.strip() + " | " + t.fillna("").astype(str).str.strip()).str.strip(" |")

        area_eff = d0[col_area] if (col_area and col_area in d0.columns) else pd.Series([""] * len(d0), index=d0.index)

        out = pd.DataFrame({
            "area": area_eff,
            "cadastro": d0[col_cad],
            "nome": d0[col_nome] if (col_nome and col_nome in d0.columns) else "",
            "dir": d0[col_dir] if (col_dir and col_dir in d0.columns) else "",
            "planta": d0[col_planta] if (col_planta and col_planta in d0.columns) else "",
            "coletor": coletor_eff,
            "data": d0[col_data] if (col_data and col_data in d0.columns) else "",
            "hora": d0[col_hora] if (col_hora and col_hora in d0.columns) else "",
            "descricao_coletor": desc_eff,
            "cargo": "",
            "_src_prio": 1,
        })
        chunks.append(out)

    if not chunks:
        return pd.DataFrame(columns=["area", "cadastro", "nome", "dir", "planta", "coletor", "descricao_coletor", "data", "hora", "cargo", "_src_prio"])
    return pd.concat(chunks, ignore_index=True)


# 0.1) Descobre cadastros do recorte (sem carregar tudo)
lt_prefixo_set = _collect_cadastros_prefixo_area_ronda(PARQUET_RONDA, prefixo_area)

# garante inclusão do cadastro forçado, mesmo fora do B-ROT* / mesmo se B-ROT* vier vazio
_c_forcado = _norm_cadastro(pd.Series([CADASTRO_FORCADO])).iloc[0]
if _c_forcado:
    lt_prefixo_set.add(_c_forcado)

# só falha se nem o recorte nem o cadastro forçado existirem
if not lt_prefixo_set:
    raise RuntimeError(
        f"Nenhum cadastro encontrado em áreas {prefixo_area}* e o cadastro forçado está vazio/ inválido."
    )

# 0.2) Carrega só os registros desses cadastros (dos dois parquets)
ronda_base = _load_ronda_filtrado(PARQUET_RONDA, lt_prefixo_set)
dados_base = _load_dados_filtrado(PARQUET_DADOS, lt_prefixo_set)

# mapa de área vindo do df_ronda.parquet + força do cadastro 2069687
area_map = _build_area_map_from_ronda(ronda_base, prefixo_area)
area_map[_c_forcado] = "B-ROT-1"

# 0.3) Concat (ronda primeiro = prioridade natural, sem sort gigante)
df_ronda_base = pd.concat([ronda_base, dados_base], ignore_index=True)

del ronda_base, dados_base
gc.collect()

# 0.4) limpeza comum (evitar cópias gigantes)
df_ronda_base["cadastro"] = _norm_cadastro(df_ronda_base["cadastro"])
df_ronda_base = df_ronda_base[df_ronda_base["cadastro"].ne("")]

df_ronda_base["area"] = df_ronda_base["area"].fillna("").astype(str).str.strip()

# propaga/sobrescreve "area" para TODOS os registros via df_ronda.parquet
_area_ronda = df_ronda_base["cadastro"].map(area_map).fillna("")
m_area = _area_ronda.ne("")
df_ronda_base.loc[m_area, "area"] = _area_ronda.loc[m_area]
df_ronda_base.loc[df_ronda_base["cadastro"].eq(_c_forcado), "area"] = "B-ROT-1"

df_ronda_base["nome"] = _fix_mojibake(df_ronda_base["nome"] if "nome" in df_ronda_base.columns else pd.Series([""] * len(df_ronda_base)))
df_ronda_base["dir"] = df_ronda_base["dir"].fillna("").astype(str)
df_ronda_base["descricao_coletor"] = _fix_mojibake(df_ronda_base["descricao_coletor"]).fillna("").astype(str)
df_ronda_base["cargo"] = df_ronda_base["cargo"].fillna("").astype(str)

df_ronda_base["planta"] = pd.to_numeric(df_ronda_base["planta"], errors="coerce").astype("Int32")
df_ronda_base["coletor"] = pd.to_numeric(df_ronda_base["coletor"], errors="coerce").astype("Int32")

df_ronda_base["data"] = pd.to_datetime(df_ronda_base["data"], errors="coerce", dayfirst=True)
df_ronda_base["hora"] = _normalize_hora_col(df_ronda_base["hora"])

m_data = df_ronda_base["data"].notna()
if not bool(m_data.all()):
    df_ronda_base = df_ronda_base.loc[m_data].copy()

df_ronda_base = df_ronda_base.drop_duplicates(
    subset=["cadastro", "data", "hora", "dir", "planta", "coletor", "descricao_coletor", "area"],
    keep="first"
)

df_ronda_base = df_ronda_base[["area", "cadastro", "nome", "dir", "planta", "coletor", "descricao_coletor", "data", "hora", "cargo"]].copy()


# ============================================================
# 1) Recorte (usa lt_prefixo_set direto + cadastro forçado)
# ============================================================
df_work = df_ronda_base[df_ronda_base["cadastro"].isin(lt_prefixo_set)].copy()
df_work = df_work.loc[:, ["area", "cadastro", "nome", "dir", "planta", "coletor", "descricao_coletor", "data", "hora", "cargo"]].copy()

df_work["cadastro"] = _norm_cadastro(df_work["cadastro"])
df_work = df_work[df_work["cadastro"].ne("")].copy()


# ============================================================
# 2) Preparo: datetime do evento + flags (SEM usar coluna "coletor")
# ============================================================
df_work["data"] = pd.to_datetime(df_work["data"], errors="coerce", dayfirst=True)
df_work = df_work.dropna(subset=["data"]).copy()

df_work["_dia_key"] = df_work["data"].dt.normalize()
df_work["hora"] = _normalize_hora_col(df_work["hora"])
df_work["_hora_td"] = pd.to_timedelta(df_work["hora"].fillna("00:00:00"), errors="coerce")
df_work["_dt_evento"] = df_work["_dia_key"] + df_work["_hora_td"]
df_work = df_work.dropna(subset=["_dt_evento"]).copy()

df_work["_dir_norm"] = _norm_dir(df_work["dir"])

desc = df_work["descricao_coletor"].fillna("").astype(str)
df_work["_is_portaria"] = desc.str.contains(regex_portaria, case=False, na=False, regex=True)

# --- SOMENTE planta ---
planta_num = pd.to_numeric(df_work["planta"], errors="coerce")

df_work["_is_999"] = planta_num.eq(999)

# presencial = planta conhecida e diferente de 999
df_work["_presencial"] = planta_num.notna() & (~planta_num.eq(999))

df_work["_is_ponto_marca"] = (
    desc.str.contains(r"(?:PONTO|MARCA)", case=False, na=False, regex=True)
    & (~df_work["_is_portaria"])
)

df_work = df_work[df_work["_is_portaria"] | df_work["_is_999"] | df_work["_presencial"]].copy()

df_work["_dia_operacional"] = (df_work["_dt_evento"] - pd.Timedelta(hours=hora_corte_operacional)).dt.normalize()
df_work = df_work.sort_values(["cadastro", "_dt_evento"], kind="mergesort")


# ============================================================
# 3) Ajuste dia operacional via pareamento Portaria E -> S
# e "cola" 999 até 6h antes/depois
# ============================================================
for cad, g_idx in df_work.groupby("cadastro", sort=False).groups.items():
    idx = pd.Index(g_idx)
    g = df_work.loc[idx, ["_dt_evento", "_dir_norm", "_is_portaria", "_is_999"]].copy()

    m_ent = g["_is_portaria"] & g["_dir_norm"].eq("E")
    m_sai = g["_is_portaria"] & g["_dir_norm"].eq("S")

    ent_times = g.loc[m_ent, "_dt_evento"].tolist()
    sai_times = g.loc[m_sai, "_dt_evento"].tolist()

    if not ent_times or not sai_times:
        continue

    pairs = []
    i = j = 0
    while i < len(ent_times) and j < len(sai_times):
        dt_ent = ent_times[i]
        dt_sai = sai_times[j]

        if dt_sai < dt_ent:
            j += 1
            continue

        if (dt_sai - dt_ent) > t_max_jornada_port:
            i += 1
            continue

        pairs.append((dt_ent, dt_sai))
        i += 1
        j += 1

    if not pairs:
        continue

    dt_series = df_work.loc[idx, "_dt_evento"]
    is_999_series = df_work.loc[idx, "_is_999"]

    for dt_ent, dt_sai in pairs:
        op_day = dt_ent.normalize()

        mask_core = (dt_series >= dt_ent) & (dt_series <= dt_sai)
        mask_999_before = is_999_series & (dt_series >= (dt_ent - t_jan_999)) & (dt_series < dt_ent)
        mask_999_after = is_999_series & (dt_series > dt_sai) & (dt_series <= (dt_sai + t_jan_999))

        df_work.loc[idx[mask_core | mask_999_before | mask_999_after], "_dia_operacional"] = op_day


# ============================================================
# 4) Calcula Irregularidade - 999 como Trajeto por (cadastro, dia_operacional)
# ============================================================
minmax = df_work.groupby("cadastro")["_dt_evento"].agg(["min", "max"]).rename(columns={"min": "_min_dt", "max": "_max_dt"})

grp = df_work.groupby(["cadastro", "_dia_operacional"], sort=False)

day = grp.agg(
    has_pres=("_presencial", "any"),
    has_999=("_is_999", "any"),
    has_port=("_is_portaria", "any"),  # mantido (não usado na condição)
).reset_index()

pres_only = df_work[df_work["_presencial"]].copy()
pres_day = (
    pres_only.groupby(["cadastro", "_dia_operacional"], sort=False)["_dt_evento"]
    .agg(first_any_pres="min", last_any_pres="max")
    .reset_index()
)

day = day.merge(pres_day, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(minmax.reset_index(), on="cadastro", how="left")

day[col_irregularidade] = False

for i, r in day.iterrows():
    # NÃO exigir portaria para marcar a irregularidade.
    if not (bool(r["has_pres"]) and bool(r["has_999"])):
        continue

    first_any = r.get("first_any_pres", pd.NaT)
    last_any = r.get("last_any_pres", pd.NaT)
    if pd.isna(first_any) or pd.isna(last_any):
        continue

    min_dt = r.get("_min_dt", pd.NaT)
    max_dt = r.get("_max_dt", pd.NaT)
    if pd.notna(min_dt) and pd.notna(max_dt):
        if (first_any - t_jan_999) < min_dt or (last_any + t_jan_999) > max_dt:
            continue

    cad = r["cadastro"]
    dia_op = r["_dia_operacional"]

    g_day = df_work[(df_work["cadastro"] == cad) & (df_work["_dia_operacional"] == dia_op)]
    p999_times = g_day.loc[g_day["_is_999"], "_dt_evento"].sort_values()
    if p999_times.empty:
        continue

    p999_before = p999_times[(p999_times >= (first_any - t_jan_999)) & (p999_times < first_any)]
    p999_after = p999_times[(p999_times > last_any) & (p999_times <= (last_any + t_jan_999))]

    odd_before = (len(p999_before) % 2 == 1) and (len(p999_before) > 0)
    odd_after = (len(p999_after) % 2 == 1) and (len(p999_after) > 0)

    if odd_before or odd_after:
        day.at[i, col_irregularidade] = True


# ============================================================
# 4.1) CORREÇÃO "FALSO VERDADEIRO" (ponto fechado) -> zera o dia
# ============================================================
def _dia_e_falso_verdadeiro(g: pd.DataFrame) -> bool:
    g = g.sort_values("_dt_evento")
    port = g[g["_is_portaria"]]
    if port.empty:
        return False

    first_port = port.iloc[0]
    last_port = port.iloc[-1]

    if str(first_port["dir"]).upper() != "E":
        return False
    if str(last_port["dir"]).upper() != "S":
        return False

    t0 = first_port["_dt_evento"]
    t1 = last_port["_dt_evento"]
    if pd.isna(t0) or pd.isna(t1) or t1 <= t0:
        return False

    # Se houver 999 válido na janela antes/depois da portaria, não descartar como falso verdadeiro.
    is_999 = g.get("_is_999", pd.Series(False, index=g.index)).fillna(False)
    if is_999.any():
        t999 = g.loc[is_999, "_dt_evento"]
        has_999_before = ((t999 >= (t0 - t_jan_999)) & (t999 < t0)).any()
        has_999_after = ((t999 > t1) & (t999 <= (t1 + t_jan_999))).any()
        if has_999_before or has_999_after:
            return False

    pontos_entre = g[
        g["_is_ponto_marca"]
        & (g["_dt_evento"] >= t0)
        & (g["_dt_evento"] <= t1)
    ]
    return len(pontos_entre) >= 2

keys_true = day.loc[day[col_irregularidade], ["cadastro", "_dia_operacional"]].drop_duplicates()
if not keys_true.empty:
    df_check = df_work.merge(keys_true, on=["cadastro", "_dia_operacional"], how="inner")
    df_check["_merge_dia"] = df_check["_dia_key"]

    keys_true2 = keys_true.copy()
    keys_true2["_merge_dia"] = keys_true2["_dia_operacional"]

    df_check = df_check.merge(
        keys_true2[["cadastro", "_merge_dia"]].drop_duplicates(),
        on=["cadastro", "_merge_dia"],
        how="inner",
    )

    grp_flag = df_check.groupby(["cadastro", "_merge_dia"], sort=False).apply(_dia_e_falso_verdadeiro)
    grupos_descartar = set(grp_flag[grp_flag].index.tolist())

    if grupos_descartar:
        day["_grp_key"] = list(zip(day["cadastro"], day["_dia_operacional"]))
        day.loc[day["_grp_key"].isin(grupos_descartar), col_irregularidade] = False
        day.drop(columns=["_grp_key"], inplace=True, errors="ignore")


# ============================================================
# 5) Base pessoa-dia para métricas e gráficos
# ============================================================
df_pessoa_dia = day[["cadastro", "_dia_operacional", col_irregularidade]].copy()
df_pessoa_dia = df_pessoa_dia.rename(columns={"_dia_operacional": "_dia_ocorrencia", col_irregularidade: "ocorreu_irregularidade_999"})
df_pessoa_dia["atividade_dia"] = 1
df_pessoa_dia["cadastro"] = _norm_cadastro(df_pessoa_dia["cadastro"])

df_pessoa_dia["_dia_ocorrencia"] = pd.to_datetime(df_pessoa_dia["_dia_ocorrencia"], errors="coerce")
df_pessoa_dia["_dia_irregularidade"] = (
    df_pessoa_dia["_dia_ocorrencia"]
    .where(df_pessoa_dia["ocorreu_irregularidade_999"])
    .dt.normalize()
)

cad_universo = pd.Index(
    sorted(_norm_cadastro(pd.Series(list(lt_prefixo_set))).unique()),
    name="cadastro"
)

nome_map = (
    df_ronda_base[["cadastro", "nome"]]
    .dropna()
    .drop_duplicates(subset=["cadastro"])
    .assign(cadastro=lambda d: _norm_cadastro(d["cadastro"]))
    .set_index("cadastro")["nome"]
    .to_dict()
)

irregularidades_por_pessoa = (
    df_pessoa_dia.groupby("cadastro")["ocorreu_irregularidade_999"]
    .sum()
    .reindex(cad_universo, fill_value=0)
    .sort_values(ascending=False)
)

pessoas_com_registro = irregularidades_por_pessoa[irregularidades_por_pessoa > 0].index.tolist()
pessoas_sem_registro = irregularidades_por_pessoa[irregularidades_por_pessoa == 0].index.tolist()

if max_pessoas_graficos is not None:
    pessoas_com_registro = pessoas_com_registro[:max_pessoas_graficos]

df_com = pd.DataFrame({"cadastro": pessoas_com_registro})
df_com["cadastro"] = _norm_cadastro(df_com["cadastro"])
df_com["nome"] = df_com["cadastro"].map(nome_map).fillna("")
df_com["qtd_irregularidades_999"] = df_com["cadastro"].map(irregularidades_por_pessoa).fillna(0).astype(int)
df_com = df_com.sort_values(["qtd_irregularidades_999", "cadastro"], ascending=[False, True])

df_sem = pd.DataFrame({"cadastro": pessoas_sem_registro})
df_sem["cadastro"] = _norm_cadastro(df_sem["cadastro"])
df_sem["nome"] = df_sem["cadastro"].map(nome_map).fillna("")
df_sem = df_sem.sort_values("cadastro")


# ============================================================
# 6) Agregado área (semanal)
# ============================================================
agg_area = (
    df_pessoa_dia.groupby(pd.Grouper(key="_dia_ocorrencia", freq=freq))
    .agg(
        total_atividades=("atividade_dia", "sum"),
        qtd_irregularidades_999=("ocorreu_irregularidade_999", "sum"),
        dias_com_irregularidade_999=("_dia_irregularidade", "nunique"),
    )
    .reset_index()
    .rename(columns={"_dia_ocorrencia": "data"})
    .sort_values("data")
)

agg_area["taxa_irregularidade_999"] = np.where(
    agg_area["total_atividades"] > 0,
    agg_area["qtd_irregularidades_999"] / agg_area["total_atividades"],
    0.0
)


# ============================================================
# 7) Métricas
# ============================================================
total_atividades = int(agg_area["total_atividades"].sum())
total_irregularidades = int(agg_area["qtd_irregularidades_999"].sum())
taxa_geral = (total_irregularidades / total_atividades) if total_atividades else 0.0

dt_ini = df_pessoa_dia["_dia_ocorrencia"].min()
dt_fim = df_pessoa_dia["_dia_ocorrencia"].max()

qtd_pessoas_recorte = int(irregularidades_por_pessoa.shape[0])
qtd_pessoas_com = int(len(pessoas_com_registro))
qtd_pessoas_sem = int(len(pessoas_sem_registro))

# >>> ALTERAÇÃO MÍNIMA: adiciona % do total de ocorrências (área) para a tabela dos piores (Top 10 pessoas)
top_pessoas = df_com[["cadastro", "nome", "qtd_irregularidades_999"]].copy()
top_pessoas["qtd_irregularidades_999"] = top_pessoas["qtd_irregularidades_999"].fillna(0).astype(int)
top_pessoas["pct_total_ocorrencias_area"] = np.where(
    total_irregularidades > 0,
    top_pessoas["qtd_irregularidades_999"] / total_irregularidades,
    0.0
)
top_10 = top_pessoas.head(10).copy()
top_10_share = (top_10["qtd_irregularidades_999"].sum() / total_irregularidades) if total_irregularidades else 0.0

top_periodos_10 = (
    agg_area[["data", "qtd_irregularidades_999", "total_atividades", "taxa_irregularidade_999"]]
    .sort_values("qtd_irregularidades_999", ascending=False)
    .head(10)
    .copy()
)
top_periodos_10["data"] = top_periodos_10["data"].dt.strftime("%Y-%m-%d")
top_periodos_10["taxa_irregularidade_999"] = (top_periodos_10["taxa_irregularidade_999"] * 100).round(2).astype(str).str.replace(".", ",") + "%"


# ============================================================
# 8) Gráficos
# ============================================================
buf_area = plot_linha_para_buffer(
    agg_area["data"],
    agg_area["qtd_irregularidades_999"],
    f"Áreas '{prefixo_area}*' — Tendência de ocorrência (Irregularidade 999)",
    "Ocorrências (pessoa-dia) no período",
    ymax=5,
)

bufs_pessoas = []
for cad in pessoas_com_registro:
    df_pdia = df_pessoa_dia[df_pessoa_dia["cadastro"] == cad].copy()

    agg_p = (
        df_pdia.groupby(pd.Grouper(key="_dia_ocorrencia", freq=freq))
        .agg(
            total_atividades=("atividade_dia", "sum"),
            qtd_irregularidades_999=("ocorreu_irregularidade_999", "sum"),
            dias_com_irregularidade_999=("_dia_irregularidade", "nunique"),
        )
        .reset_index()
        .rename(columns={"_dia_ocorrencia": "data"})
        .sort_values("data")
    )

    if agg_p["qtd_irregularidades_999"].sum() == 0:
        continue

    nome = nome_map.get(cad, "")

    buf = plot_linha_para_buffer(
        agg_p["data"],
        agg_p["qtd_irregularidades_999"],
        f"Tendência de ocorrências (Irregularidade 999) — {cad} — {nome}".strip(" —"),
        "Ocorrências (pessoa-dia) no período",
        ymax=5,
    )
    bufs_pessoas.append(buf)

# ============================================================
# 8.1) NOVO — Top 2 pessoas (ÚLTIMA página): linha suave, sem linhas pontilhadas, agregado mensal
# ============================================================
top2_cads = df_com["cadastro"].head(2).tolist()

bufs_top2 = []
for cad in top2_cads:
    df_pdia = df_pessoa_dia[df_pessoa_dia["cadastro"] == cad].copy()

    agg_p = (
        df_pdia.groupby(pd.Grouper(key="_dia_ocorrencia", freq=freq_top2))
        .agg(
            total_atividades=("atividade_dia", "sum"),
            qtd_irregularidades_999=("ocorreu_irregularidade_999", "sum"),
            dias_com_irregularidade_999=("_dia_irregularidade", "nunique"),
        )
        .reset_index()
        .rename(columns={"_dia_ocorrencia": "data"})
        .sort_values("data")
    )

    if agg_p["qtd_irregularidades_999"].sum() == 0:
        continue

    nome = nome_map.get(cad, "")

    buf = plot_linha_para_buffer(
        agg_p["data"],
        agg_p["qtd_irregularidades_999"],
        f"TOP — Tendência mensal (Irregularidade 999) — {cad} — {nome}".strip(" —"),
        "Ocorrências (pessoa-dia) por mês",
        ymax=5,
        somente_linha=True,
    )
    bufs_top2.append(buf)


# ============================================================
# 9) PDF
# ============================================================
base_styles = getSampleStyleSheet()

style_title = ParagraphStyle(
    "CORP_Title",
    parent=base_styles["Title"],
    fontName=FONT_BOLD,
    fontSize=18,
    leading=22,
    spaceAfter=8,
)

style_h1 = ParagraphStyle(
    "CORP_H1",
    parent=base_styles["Heading1"],
    fontName=FONT_BOLD,
    fontSize=14,
    leading=18,
    spaceBefore=10,
    spaceAfter=6,
)

style_h2 = ParagraphStyle(
    "CORP_H2",
    parent=base_styles["Heading2"],
    fontName=FONT_BOLD,
    fontSize=12,
    leading=15,
    spaceBefore=8,
    spaceAfter=6,
)

style_body = ParagraphStyle(
    "CORP_Body",
    parent=base_styles["BodyText"],
    fontName=FONT_REG,
    fontSize=10,
    leading=13,
)

def _header_footer(canvas: Canvas, doc):
    canvas.saveState()
    canvas.setFont(FONT_REG, 9)

    canvas.drawString(
        doc.leftMargin,
        A4[1] - 1.1 * cm,
        f"Relatório Analítico ({data_geracao_relatorio}) | {col_irregularidade} | Áreas {prefixo_area}*"
    )

    canvas.setStrokeColor(colors.lightgrey)
    canvas.line(doc.leftMargin, 1.3 * cm, A4[0] - doc.rightMargin, 1.3 * cm)

    canvas.drawRightString(A4[0] - doc.rightMargin, 0.8 * cm, f"Página {canvas.getPageNumber()}")
    canvas.restoreState()

doc = SimpleDocTemplate(
    str(pdf_path),
    pagesize=A4,
    leftMargin=1.6 * cm,
    rightMargin=1.6 * cm,
    topMargin=1.8 * cm,
    bottomMargin=1.6 * cm,
)

story = []
story.append(Paragraph("RELATÓRIO ANALÍTICO — MONITORAMENTO DE OCORRÊNCIAS", style_title))
story.append(Spacer(1, 6))

ident_html = f"""
<b>Assunto:</b> Análise de tendência — indicador “{col_irregularidade}”<br/>
<b>Escopo:</b> colaboradores associados a áreas com prefixo “{prefixo_area}*”<br/>
<b>Período analisado:</b> {dt_ini.date() if pd.notna(dt_ini) else "-"} a {dt_fim.date() if pd.notna(dt_fim) else "-"}<br/>
<b>Granularidade temporal:</b> Semanal (geral) | Mensal (Top 2)<br/>
<b>Fonte:</b> dados.parquet + df_ronda.parquet<br/>
<b>Data de geração:</b> {data_geracao_relatorio}<br/>
"""
story.append(Paragraph(ident_html, style_body))
story.append(Spacer(1, 14))

story.append(Paragraph("1. Objetivo", style_h1))
story.append(Paragraph(
    f"Apresentar a evolução temporal do indicador <b>“{col_irregularidade}”</b> no recorte das áreas <b>{prefixo_area}*</b>, "
    f"incluindo consolidação e evidências individuais para priorização de revisões.",
    style_body
))
story.append(Spacer(1, 10))

story.append(Paragraph("2. Metodologia (resumo)", style_h1))
story.append(Paragraph(
    f"- Consolidação distinta por dia (pessoa-dia).<br/>"
    f"- Agrupamento por período ({freq}) com totais e taxa.<br/>"
    f"- Correção “ponto fechado” para reduzir falsos positivos.",
    style_body
))
story.append(Spacer(1, 12))

story.append(Paragraph("3. Resultados Consolidados", style_h1))
story.append(Paragraph(
    f"<b>Total de atividades:</b> {br_int(total_atividades)}<br/>"
    f"<b>Total de ocorrências (True):</b> {br_int(total_irregularidades)}<br/>"
    f"<b>Taxa geral:</b> {br_pct(taxa_geral)}<br/>"
    f"<b>Colaboradores com ocorrência:</b> {br_int(qtd_pessoas_com)} "
    f"({br_pct(qtd_pessoas_com / qtd_pessoas_recorte) if qtd_pessoas_recorte else '0,00%'})<br/>"
    f"<b>Colaboradores sem ocorrência:</b> {br_int(qtd_pessoas_sem)} "
    f"({br_pct(qtd_pessoas_sem / qtd_pessoas_recorte) if qtd_pessoas_recorte else '0,00%'})<br/>"
    f"<b>Concentração (Top 10 pessoas):</b> {br_pct(top_10_share)} do total de ocorrências",
    style_body
))
story.append(Spacer(1, 8))

story.append(PageBreak())
story.append(Paragraph("3.1 Evidência consolidada — tendência (Área B-ROT*)", style_h2))
story.append(Spacer(1, 6))
story.append(Image(buf_area, width=16.8 * cm, height=9.0 * cm))
story.append(Spacer(1, 10))

story.append(Paragraph("4. Tabelas de Apoio", style_h1))

story.append(Paragraph("4.1 Top 10 colaboradores por volume de ocorrência", style_h2))
# >>> ALTERAÇÃO MÍNIMA: formata a coluna de percentual na tabela
top_10_tbl = top_10.copy()
top_10_tbl["pct_total_ocorrencias_area"] = top_10_tbl["pct_total_ocorrencias_area"].map(br_pct)

data_top = [["Cadastro", "Nome", "Qtd ocorrências", "Percentual"]] + top_10_tbl.values.tolist()
tbl_top = Table(data_top, repeatRows=1, colWidths=[3.0 * cm, 9.6 * cm, 3.0 * cm, 2.2 * cm])
tbl_top.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
    ("GRID", (0, 0), (-1, -1), 0.25, colors.grey),
    ("FONTNAME", (0, 0), (-1, 0), FONT_BOLD),
    ("FONTNAME", (0, 1), (-1, -1), FONT_REG),
    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ("ALIGN", (2, 1), (3, -1), "RIGHT"),
    ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.whitesmoke, colors.lightcyan]),
]))
story.append(tbl_top)
story.append(Spacer(1, 10))

story.append(Paragraph("4.2 Top 10 períodos por volume de ocorrência", style_h2))
data_tp = [["Período", "Qtd ocorrências", "Total atividades", "Taxa"]] + top_periodos_10.values.tolist()
tbl_tp = Table(data_tp, repeatRows=1, colWidths=[3.2 * cm, 3.2 * cm, 4.2 * cm, 2.2 * cm])
tbl_tp.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
    ("GRID", (0, 0), (-1, -1), 0.25, colors.grey),
    ("FONTNAME", (0, 0), (-1, 0), FONT_BOLD),
    ("FONTNAME", (0, 1), (-1, -1), FONT_REG),
    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ("ALIGN", (1, 1), (-1, -1), "RIGHT"),
    ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.whitesmoke, colors.lightcyan]),
]))
story.append(tbl_tp)

story.append(PageBreak())
story.append(Paragraph("ANEXO A — Evidências Individuais (colaboradores com ocorrência)", style_h1))
story.append(Paragraph(
    "Gráficos por colaborador (ocorrências por período), apresentados apenas quando há ocorrência no intervalo analisado.",
    style_body
))
story.append(Spacer(1, 8))

img_w = 8.3 * cm
img_h = 4.8 * cm

rows = []
row = []
for buf in bufs_pessoas:
    row.append(Image(buf, width=img_w, height=img_h))
    if len(row) == 2:
        rows.append(row)
        row = []
if row:
    row.append(Spacer(1, img_h))
    rows.append(row)

for r in rows:
    t = Table([r], colWidths=[img_w, img_w])
    t.setStyle(TableStyle([
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("LEFTPADDING", (0, 0), (-1, -1), 2),
        ("RIGHTPADDING", (0, 0), (-1, -1), 2),
        ("TOPPADDING", (0, 0), (-1, -1), 2),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
    ]))
    story.append(t)

story.append(Paragraph("<br/>ANEXO B — Colaboradores sem ocorrência no período (0 registros)", style_h1))
story.append(Spacer(1, 6))

sem_df = df_sem[["cadastro", "nome"]].copy()

chunk_size = 60
for i in range(0, len(sem_df), chunk_size):
    chunk = sem_df.iloc[i:i + chunk_size]
    data_sem = [["Cadastro", "Nome"]] + chunk.values.tolist()
    tbl_sem = Table(data_sem, repeatRows=1, colWidths=[3.0 * cm, 13.2 * cm])
    tbl_sem.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("GRID", (0, 0), (-1, -1), 0.25, colors.grey),
        ("FONTNAME", (0, 0), (-1, 0), FONT_BOLD),
        ("FONTNAME", (0, 1), (-1, -1), FONT_REG),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.whitesmoke, colors.lightcyan]),
    ]))
    story.append(tbl_sem)
    story.append(Spacer(1, 10))
    if i + chunk_size < len(sem_df):
        story.append(PageBreak())

# ============================================================
# ÚLTIMA página: Top 2 pessoas com mais ocorrência (mensal) — somente linha suave
# ============================================================
story.append(PageBreak())
story.append(Paragraph("ANEXO C — Top 2 colaboradores com mais ocorrência (tendência mensal — linha suave)", style_h1))
story.append(Paragraph(
    "Gráficos agregados por mês, exibidos apenas como linha contínua (sem marcações adicionais).",
    style_body
))
story.append(Spacer(1, 8))

if len(bufs_top2) >= 2:
    t = Table(
        [[
            Image(bufs_top2[0], width=img_w, height=img_h),
            Image(bufs_top2[1], width=img_w, height=img_h),
        ]],
        colWidths=[img_w, img_w],
    )
    t.setStyle(TableStyle([
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("LEFTPADDING", (0, 0), (-1, -1), 2),
        ("RIGHTPADDING", (0, 0), (-1, -1), 2),
        ("TOPPADDING", (0, 0), (-1, -1), 2),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 2),
    ]))
    story.append(t)

elif len(bufs_top2) == 1:
    story.append(Image(bufs_top2[0], width=16.8 * cm, height=9.0 * cm))

else:
    story.append(Paragraph("Não há colaboradores com ocorrência para compor os gráficos Top 2 no período.", style_body))

doc.build(story, onFirstPage=_header_footer, onLaterPages=_header_footer)
print(f"✅ PDF salvo.")


# ============================================================
# 10) Excel: evidências por DIA OPERACIONAL com irregularidade True
# ============================================================
keys_true = day.loc[day[col_irregularidade], ["cadastro", "_dia_operacional"]].drop_duplicates()

if keys_true.empty:
    print("⚠️ Nenhuma ocorrência True para exportar no Excel.")
else:
    df_ronda_irregularidade = df_work.merge(keys_true, on=["cadastro", "_dia_operacional"], how="inner").copy()

    mask = df_ronda_irregularidade["descricao_coletor"].astype(str).str.contains(
        r"(?:PONTO|MARCA|PORTARIA|PORT)",
        case=False, na=False, regex=True
    )
    df_ronda_irregularidade = df_ronda_irregularidade[mask].copy()

    df_ronda_irregularidade = df_ronda_irregularidade.sort_values(["cadastro", "_dt_evento"], kind="mergesort")
    out_cols = ["area", "cadastro", "nome", "cargo", "_dia_operacional", "data", "hora", "dir", "planta", "coletor", "descricao_coletor"]
    df_ronda_irregularidade = df_ronda_irregularidade[out_cols].copy()

    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        df_ronda_irregularidade.to_excel(writer, index=False)
        ws = writer.sheets["Sheet1"]

        date_col_idx = df_ronda_irregularidade.columns.get_loc("data") + 1
        for r in range(2, len(df_ronda_irregularidade) + 2):
            ws.cell(row=r, column=date_col_idx).number_format = "DD/MM/YYYY"

    print(f"✅ Excel salvo.")